# Lecture 6: Merkle Transparency Logs

Transparency logs make signed statements auditable. PACTA uses an RFC 9162-style Merkle accumulator over signed proof-check attestations. A provider signs the tree head, and an agent verifies an inclusion proof before acting on the attestation.


## Learning Objectives

- Implement leaf and node hashing with domain separation.
- Compute a Merkle root.
- Generate and verify inclusion proofs.
- Generate and verify consistency proofs.
- Explain Signed Tree Heads and signature policy.
- Explain why ML-DSA must fail closed when unavailable.


## RFC 9162 Hash Shape

PACTA follows the Certificate Transparency hash structure:

- Empty tree hash: `SHA256("")`
- Leaf hash: `SHA256(0x00 || leaf_input)`
- Node hash: `SHA256(0x01 || left || right)`

The prefix bytes prevent a leaf value from being confused with an internal node value.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "pacta").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from pacta.transparency import (
    leaf_hash,
    node_hash,
    merkle_root,
    inclusion_proof,
    verify_inclusion,
    consistency_proof,
    verify_consistency,
)

leaves = [f"attestation-{i}".encode() for i in range(1, 6)]
root = merkle_root(leaves)
print(root.hex())


In [ ]:
for index, leaf in enumerate(leaves):
    proof = inclusion_proof(leaves, index)
    ok = verify_inclusion(leaf, index, len(leaves), proof, root)
    print(index, ok, [node.hex()[:12] for node in proof])


## Consistency Proofs

An inclusion proof answers: "Is this leaf in this tree?"

A consistency proof answers: "Is the newer tree an append-only extension of the older tree?"

Both are needed for a monitored transparency system. Inclusion is enough for one agent to bind one attestation to one signed tree head. Consistency lets monitors detect equivocation or tree rewrites across time.


In [ ]:
old_size = 3
old_root = merkle_root(leaves[:old_size])
new_root = merkle_root(leaves)
proof = consistency_proof(leaves, old_size)
print("old:", old_root.hex())
print("new:", new_root.hex())
print("proof:", [node.hex()[:12] for node in proof])
print("consistent:", verify_consistency(old_size, len(leaves), old_root, new_root, proof))


## Signed Tree Heads

A Signed Tree Head records:

- log ID,
- tree size,
- timestamp,
- root hash,
- hash algorithm,
- signatures.

PACTA signs the canonical JSON STH payload with Ed25519 through OpenSSL. It also records an ML-DSA-65 slot. On this host, if no real ML-DSA backend is present, the slot is `unavailable`.

Policy matters:

- `require-signatures ed25519`: verify Ed25519 and allow ML-DSA to be unavailable.
- `require-signatures both`: require Ed25519 and ML-DSA verified. If ML-DSA is unavailable, fail closed.


In [ ]:
from pacta.postquantum import detect_ml_dsa

capability = detect_ml_dsa()
print(capability.available)
print(capability.backend)
print(capability.reason)


## Why Ed25519 and ML-DSA Together?

Ed25519 is useful because it is widely deployed, fast, and directly relevant to the Ed25519 proof corpus. That creates a deliberate "eat your own dogfood" loop: the proof-checking ecosystem signs evidence using a primitive whose implementation family is under formal scrutiny.

ML-DSA adds post-quantum robustness for the accumulator signature layer. But it must be a real signature, not an aspirational label. If a host lacks ML-DSA, the correct result is an explicit blocker.


## Split Views: why a receipt is not enough

Everything above verifies ONE receipt against ONE signed tree head. A malicious provider can maintain TWO trees - one shown to you, one shown to the world - and both views verify perfectly in isolation. This is EQUIVOCATION, and the defense is memory: pin every tree head you accept, and demand that every later tree head be CONSISTENT with your pin (same size -> same root; larger size -> a verified consistency proof from your pinned size; smaller size -> rollback, reject forever).

pacta implements this as a local STH pin store. Run the whole attack and its detection, napkin-size:


In [ ]:
# NAPKIN: pin a 2-leaf view, then let the log grow honestly - and then
# let a SPLIT VIEW present a different root at the pinned size.
import tempfile
from pathlib import Path as _P
from pacta.sthstore import check_sth_against_store
from pacta.transparency import consistency_proof, merkle_root, proof_to_hex

honest = [b"attestation-A", b"attestation-B", b"attestation-C"]
evil = [b"attestation-A", b"attestation-EVIL", b"attestation-C"]

with tempfile.TemporaryDirectory() as tmp:
    store = _P(tmp) / "sth-store.json"
    sth = lambda size, leaves: {
        "log_id": "demo-log", "tree_size": size,
        "root_hash": merkle_root(leaves[:size]).hex(),
        "timestamp": "2026-07-06T00:00:00Z",
    }
    print("pin:     ", check_sth_against_store(sth(2, honest), store).diagnostics[0])
    grown = check_sth_against_store(
        sth(3, honest), store,
        consistency_proof_hex=proof_to_hex(consistency_proof(honest, 2)),
    )
    print("grow:    ", grown.diagnostics[0])
    attack = check_sth_against_store(sth(3, evil), store)
    print("attack ok?", attack.ok)
    print("verdict: ", attack.diagnostics[0][:120], "...")


At real scale the same check runs on every `pacta receipt-verify --sth-store ...` and `pacta agent --sth-store ...` invocation; receipts embed a consistency anchor from the previous tree size, the provider serves proofs from arbitrary pinned sizes (`pacta_provider log-consistency --from-size N`), and `pacta_provider log-audit` is the monitor's self-check. A freshness policy (`--max-sth-age-seconds`) closes the stale-root hole: an old-but-valid tree head could hide later entries.

## Exercises

- Tamper with one leaf and show that inclusion verification fails.
- Explain why the tree head signature must cover tree size as well as root hash.
- Napkin, then real: run the split-view drill above; then initialize a real provider log (`pacta_provider log-init`), append two attestations, and verify the second receipt with `--sth-store` - watch the pin advance with a verified consistency proof.
- Why must the consistency anchor's ROOT (not just its size) be checked against the pin? Construct the lie that a size-only check would miss.
- Write a policy for when an autonomous agent should require `both` signatures.
- Research checkpoint: compare PACTA's pin store to production Certificate Transparency monitor/gossip requirements - what does gossip add that a single pin store cannot?
